In [ ]:
import json
import math
import random
from pathlib import Path
from typing import Optional, Tuple, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.autograd import Function
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))

Device: cuda
GPU   : NVIDIA GeForce RTX 4060 Ti


In [ ]:
CFG = {
    'model_name': 'indolem/indobert-base-uncased',
    'source_csv': 'datasets/PRDECT_train.csv',
    'target_csv': 'datasets/target_final.csv',
    'eval_csv': 'datasets/PRDECT_validation.csv',
    'output_dir': 'outputs',

    'epochs': 3,
    'batch_size': 32,
    'max_length': 128,
    'learning_rate': 1e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.05,
    'dropout': 0.1,
    'domain_loss_weight': 0.1,
    'max_grad_norm': 1.0,
    'seed': 42,
}

for k in ['source_csv', 'target_csv']:
    print(k, '=>', CFG[k], '| exists:', Path(CFG[k]).exists())
print('eval_csv =>', CFG['eval_csv'], '| exists:', Path(CFG['eval_csv']).exists() if CFG['eval_csv'] else False)

source_csv => datasets/PRDECT_train.csv | exists: True
target_csv => datasets/target_final.csv | exists: True
eval_csv => datasets/PRDECT_validation.csv | exists: True


In [24]:
LABEL2ID = {'negative': 0, 'positive': 1}
ID2LABEL = {0: 'negative', 1: 'positive'}

def _resolve_text_column(df: pd.DataFrame) -> str:
    for col in ['content', 'reviewContent', 'text', 'review']:
        if col in df.columns:
            return col
    raise ValueError(f'Cannot find text column. Available columns: {df.columns.tolist()}')

def _resolve_label_column(df: pd.DataFrame) -> str:
    for col in ['label', 'sentiment', 'target']:
        if col in df.columns:
            return col
    raise ValueError(f'Cannot find label column. Available columns: {df.columns.tolist()}')

def _normalize_labels(series: pd.Series) -> pd.Series:
    normalized = series.astype(str).str.strip().str.lower()
    return normalized.replace({'neg': 'negative', 'pos': 'positive'})

def clean_dataframe(df: pd.DataFrame, text_col: str, label_col: Optional[str] = None) -> pd.DataFrame:
    use_cols = [text_col] + ([label_col] if label_col else [])
    out = df[use_cols].copy()
    out = out.dropna(subset=[text_col])
    out[text_col] = out[text_col].astype(str).str.strip()
    out = out[out[text_col] != '']

    if label_col:
        out = out.dropna(subset=[label_col])
        out[label_col] = _normalize_labels(out[label_col])
        out = out[out[label_col].isin(LABEL2ID)]

    return out.reset_index(drop=True)

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128, labels=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )

        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

def make_loaders(
    tokenizer,
    source_path: Path,
    target_path: Path,
    eval_path: Optional[Path],
    batch_size: int,
    max_length: int,
    seed: int,
) -> Tuple[DataLoader, DataLoader, DataLoader, Optional[DataLoader], dict]:
    source_df = pd.read_csv(source_path)
    source_text_col = _resolve_text_column(source_df)
    source_label_col = _resolve_label_column(source_df)
    source_df = clean_dataframe(source_df, source_text_col, source_label_col)
    source_df['label_id'] = source_df[source_label_col].map(LABEL2ID)

    train_df, val_df = train_test_split(
        source_df,
        test_size=0.1,
        random_state=seed,
        stratify=source_df['label_id'],
    )

    target_df = pd.read_csv(target_path)
    target_text_col = _resolve_text_column(target_df)
    target_df = clean_dataframe(target_df, target_text_col)

    source_train_ds = TextDataset(
        texts=train_df[source_text_col].tolist(),
        labels=train_df['label_id'].tolist(),
        tokenizer=tokenizer,
        max_length=max_length,
    )
    source_val_ds = TextDataset(
        texts=val_df[source_text_col].tolist(),
        labels=val_df['label_id'].tolist(),
        tokenizer=tokenizer,
        max_length=max_length,
    )
    target_train_ds = TextDataset(
        texts=target_df[target_text_col].tolist(),
        labels=None,
        tokenizer=tokenizer,
        max_length=max_length,
    )

    source_train_loader = DataLoader(source_train_ds, batch_size=batch_size, shuffle=True)
    source_val_loader = DataLoader(source_val_ds, batch_size=batch_size, shuffle=False)
    target_train_loader = DataLoader(target_train_ds, batch_size=batch_size, shuffle=True)

    eval_loader = None
    eval_samples = 0

    if eval_path is not None and eval_path.exists():
        eval_df = pd.read_csv(eval_path)
        eval_text_col = _resolve_text_column(eval_df)
        eval_label_col = _resolve_label_column(eval_df)

        if 'origin' in eval_df.columns:
            origin = eval_df['origin'].astype(str).str.lower()
            target_only = eval_df[origin == 'target']
            if len(target_only) > 0:
                eval_df = target_only

        eval_df = clean_dataframe(eval_df, eval_text_col, eval_label_col)
        eval_df['label_id'] = eval_df[eval_label_col].map(LABEL2ID)

        eval_ds = TextDataset(
            texts=eval_df[eval_text_col].tolist(),
            labels=eval_df['label_id'].tolist(),
            tokenizer=tokenizer,
            max_length=max_length,
        )
        eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False)
        eval_samples = len(eval_ds)

    stats = {
        'source_train': len(source_train_ds),
        'source_val': len(source_val_ds),
        'target_train': len(target_train_ds),
        'target_eval': eval_samples,
    }

    return source_train_loader, source_val_loader, target_train_loader, eval_loader, stats

In [25]:
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, inputs, alpha):
        ctx.alpha = alpha
        return inputs.view_as(inputs)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

class GradientReversalLayer(nn.Module):
    def forward(self, inputs, alpha=1.0):
        return GradientReversalFunction.apply(inputs, alpha)

class DANNModel(nn.Module):
    def __init__(self, model_name: str, num_labels: int = 2, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        # self.label_classifier = nn.Linear(hidden_size, num_labels)
        self.label_classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, num_labels),
        )

        self.grl = GradientReversalLayer()
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2),
        )

    def extract_features(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0]

    def forward(self, input_ids, attention_mask, alpha=1.0):
        features = self.extract_features(input_ids, attention_mask)
        features = self.dropout(features)

        class_logits = self.label_classifier(features)

        reversed_features = self.grl(features, alpha=alpha)
        domain_logits = self.domain_classifier(reversed_features)

        return class_logits, domain_logits

In [26]:
@torch.no_grad()
def evaluate_label_head(model, dataloader, device):
    model.eval()

    all_preds, all_labels = [], []
    total_loss = 0.0
    loss_fn = nn.CrossEntropyLoss()

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        class_logits, _ = model(input_ids, attention_mask, alpha=0.0)
        loss = loss_fn(class_logits, labels)

        total_loss += loss.item()
        all_preds.extend(torch.argmax(class_logits, dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

    return {
        'loss': avg_loss,
        'acc': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'preds': all_preds,
        'labels': all_labels,
    }

def train_dann(cfg: dict):
    set_seed(cfg['seed'])

    source_path = Path(cfg['source_csv'])
    target_path = Path(cfg['target_csv'])
    eval_path = Path(cfg['eval_csv']) if cfg.get('eval_csv') else None

    tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])

    source_train_loader, source_val_loader, target_train_loader, target_eval_loader, stats = make_loaders(
        tokenizer=tokenizer,
        source_path=source_path,
        target_path=target_path,
        eval_path=eval_path,
        batch_size=cfg['batch_size'],
        max_length=cfg['max_length'],
        seed=cfg['seed'],
    )

    print('Dataset summary:', stats)

    model = DANNModel(model_name=cfg['model_name'], num_labels=2, dropout=cfg['dropout']).to(device)

    class_loss_fn = nn.CrossEntropyLoss()
    domain_loss_fn = nn.CrossEntropyLoss()

    optimizer = AdamW(model.parameters(), lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'])

    steps_per_epoch = min(len(source_train_loader), len(target_train_loader))
    total_steps = cfg['epochs'] * steps_per_epoch
    warmup_steps = int(cfg['warmup_ratio'] * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_val_f1 = -1.0
    best_dir = Path(cfg['output_dir']) / 'best7'
    best_dir.mkdir(parents=True, exist_ok=True)

    global_step = 0
    last_val_metrics = None

    for epoch in range(1, cfg['epochs'] + 1):
        model.train()

        running_total = 0.0
        running_cls = 0.0
        running_domain = 0.0

        source_iter = iter(source_train_loader)
        target_iter = iter(target_train_loader)

        progress = tqdm(range(steps_per_epoch), desc=f'Epoch {epoch}/{cfg["epochs"]}')
        for step_idx in progress:
            source_batch = next(source_iter)
            target_batch = next(target_iter)

            p = float(global_step) / max(1, total_steps - 1)
            alpha = 2.0 / (1.0 + math.exp(-10 * p)) - 1.0

            source_input_ids = source_batch['input_ids'].to(device)
            source_attention_mask = source_batch['attention_mask'].to(device)
            source_labels = source_batch['labels'].to(device)

            target_input_ids = target_batch['input_ids'].to(device)
            target_attention_mask = target_batch['attention_mask'].to(device)

            class_logits, source_domain_logits = model(source_input_ids, source_attention_mask, alpha=alpha)
            _, target_domain_logits = model(target_input_ids, target_attention_mask, alpha=alpha)

            cls_loss = class_loss_fn(class_logits, source_labels)

            source_domain_labels = torch.zeros(source_domain_logits.size(0), dtype=torch.long, device=device)
            target_domain_labels = torch.ones(target_domain_logits.size(0), dtype=torch.long, device=device)

            domain_source_loss = domain_loss_fn(source_domain_logits, source_domain_labels)
            domain_target_loss = domain_loss_fn(target_domain_logits, target_domain_labels)
            domain_loss = 0.5 * (domain_source_loss + domain_target_loss)

            total_loss = cls_loss + cfg['domain_loss_weight'] * domain_loss

            optimizer.zero_grad()
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['max_grad_norm'])
            optimizer.step()
            scheduler.step()

            running_total += total_loss.item()
            running_cls += cls_loss.item()
            running_domain += domain_loss.item()
            global_step += 1

            seen = step_idx + 1
            progress.set_postfix(
                total=f'{running_total/seen:.4f}',
                cls=f'{running_cls/seen:.4f}',
                dom=f'{running_domain/seen:.4f}',
                alpha=f'{alpha:.3f}',
            )

        val_metrics = evaluate_label_head(model, source_val_loader, device)
        last_val_metrics = val_metrics
        print(
            f'Epoch {epoch} | val_loss={val_metrics["loss"]:.4f} val_acc={val_metrics["acc"]:.4f} '
            f'val_precision={val_metrics["precision"]:.4f} val_recall={val_metrics["recall"]:.4f} val_f1={val_metrics["f1"]:.4f}'
        )

        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            model.encoder.save_pretrained(best_dir)
            tokenizer.save_pretrained(best_dir)
            torch.save(
                {
                    'label_classifier': model.label_classifier.state_dict(),
                    'domain_classifier': model.domain_classifier.state_dict(),
                    'args': cfg,
                },
                best_dir / 'dann_heads.pt',
            )
            print('  Saved new best model to:', best_dir)

    print(f'Training complete. Best source-val F1: {best_val_f1:.4f}')

    comparison_metrics = {
        'framework': 'dann',
        'split': 'source_val',
        'loss': last_val_metrics['loss'],
        'accuracy': last_val_metrics['acc'],
        'precision': last_val_metrics['precision'],
        'recall': last_val_metrics['recall'],
        'f1': last_val_metrics['f1'],
    }
    
    print("Source Evaluation")
    print("===========================")
    print(comparison_metrics)

    if target_eval_loader is not None:
        print('\nEvaluating best checkpoint on target labeled set...')
        best_model = DANNModel(model_name=str(best_dir), num_labels=2, dropout=cfg['dropout']).to(device)
        heads = torch.load(best_dir / 'dann_heads.pt', map_location=device)
        best_model.label_classifier.load_state_dict(heads['label_classifier'])
        best_model.domain_classifier.load_state_dict(heads['domain_classifier'])

        eval_metrics = evaluate_label_head(best_model, target_eval_loader, device)
        print(
            f'Target eval | loss={eval_metrics["loss"]:.4f} acc={eval_metrics["acc"]:.4f} '
            f'precision={eval_metrics["precision"]:.4f} recall={eval_metrics["recall"]:.4f} f1={eval_metrics["f1"]:.4f}'
        )
        print('\nClassification report (target eval):')
        print(classification_report(eval_metrics['labels'], eval_metrics['preds'], target_names=[ID2LABEL[0], ID2LABEL[1]], digits=4))

        # comparison_metrics = {
        #     'framework': 'dann',
        #     'split': 'target_eval',
        #     'loss': eval_metrics['loss'],
        #     'accuracy': eval_metrics['acc'],
        #     'precision': eval_metrics['precision'],
        #     'recall': eval_metrics['recall'],
        #     'f1': eval_metrics['f1'],
        # }
        target_eval_metrics = {
            'loss': eval_metrics['loss'],
            'accuracy': eval_metrics['acc'],
            'precision': eval_metrics['precision'],
            'recall': eval_metrics['recall'],
            'f1': eval_metrics['f1'],
        }
        
        comparison_metrics['target_eval'] = target_eval_metrics
        comparison_metrics['primary_split'] = 'target_eval'
        comparison_metrics['loss'] = target_eval_metrics['loss']
        comparison_metrics['accuracy'] = target_eval_metrics['accuracy']
        comparison_metrics['precision'] = target_eval_metrics['precision']
        comparison_metrics['recall'] = target_eval_metrics['recall']
        comparison_metrics['f1'] = target_eval_metrics['f1']
        
        print("Target Evaluation")
        print("===========================")
        print(target_eval_metrics)

    comparison_path = Path(cfg['output_dir']) / 'comparison_metrics.json'
    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    with open(comparison_path, 'w', encoding='utf-8') as f:
        json.dump(comparison_metrics, f, indent=2)

    print('Saved comparison metrics to:', comparison_path)
    return best_dir

best_model_dir = train_dann(CFG)
print('Best checkpoint dir:', best_model_dir)

Dataset summary: {'source_train': 3819, 'source_val': 425, 'target_train': 38341, 'target_eval': 1061}


c:\Anaconda3\envs\computer_vision\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Epoch 1/3:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 1 | val_loss=0.2011 val_acc=0.9341 val_precision=0.9341 val_recall=0.9341 val_f1=0.9341
  Saved new best model to: outputs\best7


Epoch 2/3:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 2 | val_loss=0.1002 val_acc=0.9553 val_precision=0.9554 val_recall=0.9553 val_f1=0.9553
  Saved new best model to: outputs\best7


Epoch 3/3:   0%|          | 0/120 [00:00<?, ?it/s]

Epoch 3 | val_loss=0.1005 val_acc=0.9553 val_precision=0.9553 val_recall=0.9553 val_f1=0.9553
  Saved new best model to: outputs\best7
Training complete. Best source-val F1: 0.9553
Source Evaluation
{'framework': 'dann', 'split': 'source_val', 'loss': 0.10054439179865378, 'accuracy': 0.9552941176470588, 'precision': 0.9552957883992368, 'recall': 0.9552941176470588, 'f1': 0.9552896545221601}

Evaluating best checkpoint on target labeled set...
Target eval | loss=0.1031 acc=0.9623 precision=0.9623 recall=0.9623 f1=0.9623

Classification report (target eval):
              precision    recall  f1-score   support

    negative     0.9620    0.9655    0.9637       550
    positive     0.9627    0.9589    0.9608       511

    accuracy                         0.9623      1061
   macro avg     0.9623    0.9622    0.9622      1061
weighted avg     0.9623    0.9623    0.9623      1061

Target Evaluation
{'loss': 0.1031185620573952, 'accuracy': 0.9622997172478793, 'precision': 0.9623010657567546

# Infer

In [ ]:
def load_model(model_dir: Path, heads_path: Optional[Path] = None, dropout: float = 0.1, device_override: Optional[torch.device] = None):
    if device_override is None:
        device_override = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    heads_path = heads_path or (model_dir / 'dann_heads.pt')
    if not model_dir.exists():
        raise FileNotFoundError(f'Model directory not found: {model_dir}')
    if not heads_path.exists():
        raise FileNotFoundError(f'Heads checkpoint not found: {heads_path}')

    heads = torch.load(heads_path, map_location=device_override)
    train_args = heads.get('args', {})
    model_dropout = float(train_args.get('dropout', dropout))

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = DANNModel(model_name=str(model_dir), num_labels=2, dropout=model_dropout).to(device_override)
    model.label_classifier.load_state_dict(heads['label_classifier'])
    model.domain_classifier.load_state_dict(heads['domain_classifier'])
    model.eval()

    return model, tokenizer, device_override

@torch.no_grad()
def predict_texts(model: DANNModel, tokenizer, texts: List[str], device_infer: torch.device, max_length: int = 128, batch_size: int = 16):
    rows = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoding = tokenizer(batch_texts, truncation=True, padding=True, max_length=max_length, return_tensors='pt')

        input_ids = encoding['input_ids'].to(device_infer)
        attention_mask = encoding['attention_mask'].to(device_infer)

        class_logits, domain_logits = model(input_ids, attention_mask, alpha=0.0)

        class_probs = F.softmax(class_logits, dim=1).cpu()
        domain_probs = F.softmax(domain_logits, dim=1).cpu()

        for i, text in enumerate(batch_texts):
            sentiment_id = int(torch.argmax(class_probs[i]).item())
            domain_id = int(torch.argmax(domain_probs[i]).item())
            rows.append({
                'text': text,
                'sentiment_id': sentiment_id,
                'sentiment_label': ID2LABEL[sentiment_id],
                'sentiment_negative_prob': float(class_probs[i][0].item()),
                'sentiment_positive_prob': float(class_probs[i][1].item()),
                'domain_pred_id': domain_id,
                'domain_pred_label': 'source' if domain_id == 0 else 'target',
                'domain_source_prob': float(domain_probs[i][0].item()),
                'domain_target_prob': float(domain_probs[i][1].item()),
            })

    return rows

def print_results(rows: List[dict], max_rows: int = 20):
    preview = rows[:max_rows]
    if not preview:
        print('No predictions.')
        return

    print(f'Showing {len(preview)} / {len(rows)} predictions')
    print('=' * 80)
    for i, row in enumerate(preview, 1):
        print(f'[{i}] sentiment={row["sentiment_label"]} (neg={row["sentiment_negative_prob"]:.4f}, pos={row["sentiment_positive_prob"]:.4f})')
        print(f'    domain={row["domain_pred_label"]} (source={row["domain_source_prob"]:.4f}, target={row["domain_target_prob"]:.4f})')
        print(f'    text={row["text"]}')
        print('-' * 80)

In [ ]:
model_dir = Path(CFG['output_dir']) / 'best7'

model, tokenizer, infer_device = load_model(model_dir=model_dir)
print('Inference device:', infer_device)

texts = [
    "Aplikasinya sangat membantu dan mudah digunakan!",
    "Pengiriman cepat, barang sesuai deskripsi. Sangat puas!",
    "Sangat kecewa dengan pelayanan. Tidak profesional!",
    "Aplikasi sering error dan lambat. Tolong diperbaiki.",
]

rows = predict_texts(
    model=model,
    tokenizer=tokenizer,
    texts=texts,
    device_infer=infer_device,
    max_length=CFG['max_length'],
    batch_size=CFG['batch_size'],
)

print_results(rows)
pd.DataFrame(rows)

Inference device: cuda
Showing 4 / 4 predictions
[1] sentiment=positive (neg=0.0083, pos=0.9917)
    domain=target (source=0.3651, target=0.6349)
    text=Aplikasinya sangat membantu dan mudah digunakan!
--------------------------------------------------------------------------------
[2] sentiment=positive (neg=0.0063, pos=0.9937)
    domain=target (source=0.3629, target=0.6371)
    text=Pengiriman cepat, barang sesuai deskripsi. Sangat puas!
--------------------------------------------------------------------------------
[3] sentiment=negative (neg=0.9835, pos=0.0165)
    domain=source (source=0.6829, target=0.3171)
    text=Sangat kecewa dengan pelayanan. Tidak profesional!
--------------------------------------------------------------------------------
[4] sentiment=negative (neg=0.8618, pos=0.1382)
    domain=source (source=0.6612, target=0.3388)
    text=Aplikasi sering error dan lambat. Tolong diperbaiki.
---------------------------------------------------------------------------

,text,sentiment_id,sentiment_label,sentiment_negative_prob,sentiment_positive_prob,domain_pred_id,domain_pred_label,domain_source_prob,domain_target_prob
0,Aplikasinya sangat membantu dan mudah digunakan!,1,positive,0.008268,0.991732,1,target,0.365093,0.634907
1,"Pengiriman cepat, barang sesuai deskripsi. San...",1,positive,0.006289,0.993711,1,target,0.362906,0.637094
2,Sangat kecewa dengan pelayanan. Tidak profesio...,0,negative,0.983506,0.016494,0,source,0.682924,0.317076
3,Aplikasi sering error dan lambat. Tolong diper...,0,negative,0.861781,0.138219,0,source,0.661233,0.338767
